# Data Preparation

## Tokenizer

In [1]:
import pickle
import torch

class WubiCharTokenizer:
    """
    Tokenizes Chinese text into sequences of atomic Wubi symbols (letters + separator).
    Each character becomes a fixed number of tokens (its Wubi code letters + separator).
    No merging across characters occurs.
    """
    def __init__(self, wubi_dict_path):
        with open(wubi_dict_path, 'rb') as f:
            self.ch2wubi = pickle.load(f)

        # Build vocabulary: all possible letters + separator
        self.sep = '#'
        letters = set()
        for code in self.ch2wubi.values():
            letters.update(code)
        # Also include digits? Original mapping strips digits, so just letters.
        self.vocab = ['[PAD]', '[UNK]', self.sep] + sorted(letters)
        self.stoi = {s: i for i, s in enumerate(self.vocab)}
        self.itos = {i: s for s, i in self.stoi.items()}

    def encode(self, text):
        """
        Convert text to a list of token IDs and a list of character spans.
        Returns:
            ids: list of int (token IDs)
            char_spans: list of (start, end) token indices for each character
        """
        text = text.lower()
        ids = []
        char_spans = []
        for ch in text:
            # Get Wubi code (fallback to character itself)
            code = self.ch2wubi.get(ch, ch)
            # Convert each symbol in the code to an ID
            code_ids = [self.stoi.get(c, self.stoi['[UNK]']) for c in code]
            start = len(ids)
            ids.extend(code_ids)
            ids.append(self.stoi[self.sep])
            end = len(ids)
            char_spans.append((start, end))
        return ids, char_spans

    def decode(self, ids):
        """For debugging: convert IDs back to string."""
        return ''.join(self.itos[i] for i in ids if i != self.stoi[self.sep])

In [2]:
# Load the Wubi mapping (from the SubCharTokenization repo)
wubi_dict = "/kaggle/input/notebooks/davidvista/wubi-tokenizer/chinese_to_wubi.pkl"
tokenizer = WubiCharTokenizer(wubi_dict)

text = "我是最高的学生"
ids, spans = tokenizer.encode(text)

print("Token IDs:", ids)
print("Character spans:", spans)
# Output: spans like [(0,5), (5,8), (8,15), ...] depending on code lengths

Token IDs: [29, 2, 22, 2, 22, 14, 2, 37, 25, 2, 30, 2, 21, 28, 2, 32, 19, 2]
Character spans: [(0, 2), (2, 4), (4, 7), (7, 10), (10, 12), (12, 15), (15, 18)]


In [3]:
# Inverse mapping from Wubi to Character
with open("/kaggle/input/notebooks/davidvista/wubi-tokenizer/wubi_to_chinese.pkl", 'rb') as f:
    wubi2ch = pickle.load(f)

## Pinyin Mapper

In [4]:
INITIALS = [
    "", "b","p","m","f","d","t","n","l","g","k","h",
    "j","q","x","zh","ch","sh","r","z","c","s"
]

FINALS = [
    "a","ai","an","ang","ao","e","ei","en","eng","er",
    "i","ia","ian","iang","iao","ie","in","ing","iong","iu",
    "o","ong","ou",
    "u","ua","uai","uan","uang","ui","un","uo",
    "v","ve","van","vn"
]

TONES = ["1","2","3","4","5"]

ZERO_INITIAL_MAP = {
    # y-series
    "yi":"i",
    "ya":"ia",
    "yao":"iao",
    "ye":"ie",
    "you":"iu",
    "yan":"ian",
    "yin":"in",
    "yang":"iang",
    "ying":"ing",
    "yong":"iong",

    "yu":"v",
    "yue":"ve",
    "yuan":"van",
    "yun":"vn",

    # w-series
    "wu":"u",
    "wa":"ua",
    "wo":"uo",
    "wai":"uai",
    "wei":"ui",
    "wan":"uan",
    "wen":"un",
    "wang":"uang",
    "weng":"ong",
}


def normalize_zero_initial(base):
    return ZERO_INITIAL_MAP.get(base, base)


def normalize_jqx(final):
    if final.startswith("u"):
        mapping = {
            "u": "v",
            "ue": "ve",
            "uan": "van",
            "un": "vn",
        }
        return mapping.get(final, final)
    return final


init2idx = {v:i for i,v in enumerate(INITIALS)}
final2idx = {v:i for i,v in enumerate(FINALS)}
tone2idx = {v:i for i,v in enumerate(TONES)}

idx2init = {i:v for v,i in init2idx.items()}
idx2final = {i:v for v,i in final2idx.items()}
idx2tone = {i:v for v,i in tone2idx.items()}


def pinyin_to_idx(token: str):
    tone = token[-1]

    if not tone.isdigit():
        tone = "5" # neutral tone, usually has no digit
        base = token
    else:
        base = token[:-1]

    base = normalize_zero_initial(base)

    # longest-match for initials

    try:
        for ini in sorted(INITIALS, key=len, reverse=True):
            if base.startswith(ini):
                final = base[len(ini):]

                if ini in {"j","q","x"}:
                    final = normalize_jqx(final)
                
                return [init2idx[ini], final2idx[final], tone2idx[tone]]
    except Exception:
        raise ValueError("Invalid pinyin token")

## Dataset

In [5]:
import pandas as pd

dataset = pd.read_csv("/kaggle/input/notebooks/davidvista/pinyin-dataset-labelling/chinese-short-sentences-pinyin.csv")

In [6]:
dataset.head()

,text,pinyin
0,一个特殊群体的期待在密切关注此次特金会的人中有一个特殊的群体在日本殖民期间来到日本的韩国朝鲜...,yi1 ge4 te4 shu1 qun2 ti3 de qi1 dai4 zai4 mi4...
1,他们希望政治解冻有助于让朝鲜摆脱孤立,ta1 men xi1 wang4 zheng4 zhi4 jie3 dong4 you3 ...
2,他认为对这个奖的妄想可能会促使双方做出轻率的承诺但也可能有助于艰巨的和平进程,ta1 ren4 wei2 dui4 zhe4 ge4 jiang3 de wang4 xi...
3,欢迎阅读纪思道文章的中文版,huan1 ying2 yue4 du2 ji4 si1 dao4 wen2 zhang1 ...
4,不法业者向一些中国女性编造了一个美好的未来到美国开始新生活并拥有一份合法的工作,bu4 fa3 ye4 zhe3 xiang4 yi1 xie1 zhong1 guo2 n...


In [7]:
from tqdm import tqdm 

def tokenize_dataset(dataset, tokenizer, pinyin_to_idx):

    for i, sample in tqdm(dataset.iterrows(), total=len(dataset)):
        text = sample['text']
        pinyin_str = sample['pinyin']

        char_seq = list(text)

        pinyin_tokens = pinyin_str.split()
        
        # Valid sample - yield the indices
        token_idxs, char_span = tokenizer.encode(text)

        pinyin_idxs = []
        for token in pinyin_tokens:

            try:
                pinyin_idx = pinyin_to_idx(token)
                pinyin_idxs.append(pinyin_idx)
            except ValueError:
                print(f"Skipped token {token}")
        
        yield char_seq, token_idxs, char_span, pinyin_idxs

preprocessed_dataset = tokenize_dataset(dataset, tokenizer, pinyin_to_idx)

In [8]:
char_seqs = []
token_seqs = []
char_spans = []
pinyin_seqs = []

for char_seq, token_idxs, char_span, pinyin_idxs in preprocessed_dataset:
    char_seqs.append(char_seq)
    token_seqs.append(token_idxs)
    char_spans.append(char_span)
    pinyin_seqs.append(pinyin_idxs)

100%|██████████| 161521/161521 [00:31<00:00, 5123.85it/s]


Additional step, build the index mapping to characters (as labels for the encoder):

In [9]:
def build_char_vocab(char_seqs, add_pad=True, add_unk=True):
    """
    Build character vocabulary from a list of character sequences.

    Args:
        char_seqs: list of lists of strings (each inner list is a sentence of characters)
        add_pad: if True, include <PAD> token at index 0
        add_unk: if True, include <UNK> token after pad (if pad exists) or at start

    Returns:
        char2idx: dict mapping character to index
        idx2char: list mapping index to character
    """
    # Collect all unique characters
    chars = set()
    for seq in char_seqs:
        for ch in seq:
            chars.add(ch)
    # Sort for deterministic order
    sorted_chars = sorted(chars)

    # Build vocabulary list
    vocab = []
    if add_pad:
        vocab.append('<PAD>')
    if add_unk:
        vocab.append('<UNK>')
    vocab.extend(sorted_chars)

    # Create mappings
    char2idx = {ch: idx for idx, ch in enumerate(vocab)}
    idx2char = vocab

    return char2idx, idx2char

In [10]:
char2idx, idx2char = build_char_vocab(char_seqs)

In [11]:
char_indices = []

for char_sequence in char_seqs:
    indices = [char2idx[char] for char in char_sequence]
    char_indices.append(indices)

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class PinyinDataset(Dataset):
    """Dataset for character‑to‑pinyin with sub‑character tokens, including raw characters and their indices."""
    def __init__(self, char_sequences, char_indices, token_sequences, char_spans, pinyin_sequences):
        """
        Args:
            char_sequences: list of lists of characters (strings) for each sample
            char_indices: list of lists of integer indices for each character (same length as char_sequences)
            token_sequences: list of lists of token indices (sub‑character tokens)
            char_spans: list of lists of (start, end) tuples for each character
            pinyin_sequences: list of lists of decomposed pinyin vectors (initial, final, tone)
        """
        assert len(char_sequences) == len(char_indices) == len(token_sequences) == len(char_spans) == len(pinyin_sequences)
        self.char_seqs = char_sequences
        self.char_idxs = char_indices
        self.token_seqs = token_sequences
        self.char_spans = char_spans
        self.pinyin_seqs = pinyin_sequences

    def __len__(self):
        return len(self.token_seqs)

    def __getitem__(self, idx):
        return {
            'chars': self.char_seqs[idx],
            'char_idxs': torch.tensor(self.char_idxs[idx], dtype=torch.long),
            'tokens': torch.tensor(self.token_seqs[idx], dtype=torch.long),
            'offsets': torch.tensor([span[0] for span in self.char_spans[idx]], dtype=torch.long),
            'pinyin': torch.tensor(self.pinyin_seqs[idx], dtype=torch.long)  # shape (num_chars, 3)
        }

def collate_fn(batch, pad_idx=-100):
    """
    Collate a batch: flatten tokens, build offsets, and also collect character lists and indices.
    Returns:
        flat_tokens: (total_tokens,) concatenated token IDs
        char_offsets: (total_chars,) start index of each character in flat_tokens
        char_lengths: (batch,) number of characters per sample
        padded_pinyin: (batch, max_chars, 3) padded pinyin targets
        chars_list: list of lists of characters (one per sample)
        padded_char_idxs: (batch, max_chars) padded character indices
    """
    chars_list = [item['chars'] for item in batch]
    char_idxs_list = [item['char_idxs'] for item in batch]
    tokens_list = [item['tokens'] for item in batch]
    offsets_list = [item['offsets'] for item in batch]
    pinyin_list = [item['pinyin'] for item in batch]

    flat_tokens = []
    char_offsets = []
    char_lengths = []
    total_tokens = 0

    for sample_tokens, sample_offsets in zip(tokens_list, offsets_list):
        flat_tokens.extend(sample_tokens.tolist())
        num_chars = len(sample_offsets)
        char_lengths.append(num_chars)
        for i in range(num_chars):
            char_offsets.append(total_tokens + sample_offsets[i].item())
        total_tokens += len(sample_tokens)

    flat_tokens = torch.tensor(flat_tokens, dtype=torch.long)
    char_offsets = torch.tensor(char_offsets, dtype=torch.long)
    char_lengths = torch.tensor(char_lengths, dtype=torch.long)

    padded_pinyin = nn.utils.rnn.pad_sequence(pinyin_list, batch_first=True,
                                              padding_value=pad_idx)
    padded_char_idxs = nn.utils.rnn.pad_sequence(char_idxs_list, batch_first=True,
                                                 padding_value=pad_idx)

    return flat_tokens, char_offsets, char_lengths, padded_pinyin, chars_list, padded_char_idxs

def create_dataloaders(char_sequences, char_indices, token_sequences, char_spans, pinyin_sequences, batch_size,
                       val_split=0.1, test_split=0.1, pad_idx=0, shuffle=True):
    """
    Create train/validation/test DataLoaders.
    Args:
        char_sequences: list of lists of characters (strings) per sample
        char_indices: list of lists of integer indices for each character (same length as char_sequences)
        token_sequences: list of token index sequences (per sample)
        char_spans: list of character span lists (per sample)
        pinyin_sequences: list of decomposed pinyin sequences (per sample)
        batch_size: batch size
        val_split: fraction of data for validation
        test_split: fraction of data for test
        pad_idx: padding value (e.g., -100)
        shuffle: whether to shuffle training data
    Returns:
        train_loader, val_loader, test_loader
    """
    dataset = PinyinDataset(
        char_sequences, char_indices, token_sequences, char_spans, pinyin_sequences
    )

    total = len(dataset)
    val_size = int(total * val_split)
    test_size = int(total * test_split)
    train_size = total - val_size - test_size

    assert train_size > 0, "Not enough data for training split"
    assert val_size >= 0 and test_size >= 0, "Split sizes must be non‑negative"

    lengths = [train_size, val_size, test_size]
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, lengths,
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    return train_loader, val_loader, test_loader

In [13]:
PAD_IDX = -100

train_loader, val_loader, test_loader = create_dataloaders(
    char_seqs, char_indices, token_seqs, char_spans, pinyin_seqs,
    batch_size=32, val_split=0.1, test_split=0.1, pad_idx=PAD_IDX
)

# Model Preparation

## Encoder Model

The encoder model will produce embeddings for characters to be further decoded into the original characters.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class ClassifierHead(nn.Module):
    """MLP with one hidden layer for classification."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class Encoder(nn.Module):
    """
    BiLSTM model for predicting pinyin from characters.
    - Token embeddings aggregated per character (EmbeddingBag).
    - Character representations passed through BiLSTM.
    - Three MLP heads predict initial, final, and tone.
    """
    def __init__(self, vocab_size_tokens, embedding_dim, hidden_dim,
                 num_initials, num_finals, num_tones,
                 num_layers=2, dropout=0.3, mlp_hidden_dim=128):
        super().__init__()
        self.embedding_bag = nn.EmbeddingBag(vocab_size_tokens, embedding_dim,
                                              mode='mean')
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

        # MLP heads (input = hidden_dim*2 from BiLSTM)
        self.initial_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_initials, dropout)
        self.final_head   = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_finals, dropout)
        self.tone_head    = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_tones, dropout)

    def forward(self, token_ids, token_offsets, char_lengths, return_hidden=False):
        """
        Args:
            token_ids: (total_tokens,) flattened token IDs (no padding)
            token_offsets: (total_chars,) start indices for each character
            char_lengths: (batch,) number of characters per sample
            return_hidden: if True, also return character‑level hidden states
        Returns:
            logits_initial, logits_final, logits_tone:
                each shape (batch, max_char_len, respective_vocab_size)
            (optional) char_out: (batch, max_char_len, hidden_dim*2) before heads
        """
        char_emb = self.embedding_bag(token_ids, token_offsets)  # (total_chars, emb_dim)

        # Split into per‑sample sequences and pad
        char_emb_list = torch.split(char_emb, char_lengths.tolist())
        padded_char_emb = nn.utils.rnn.pad_sequence(char_emb_list, batch_first=True)

        # Pack for LSTM
        packed_emb = pack_padded_sequence(padded_char_emb, char_lengths.cpu(),
                                          batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed_emb)
        char_out, _ = pad_packed_sequence(packed_out, batch_first=True)
        char_out = self.dropout(char_out)

        # Three MLP heads
        logits_initial = self.initial_head(char_out)
        logits_final   = self.final_head(char_out)
        logits_tone    = self.tone_head(char_out)

        if return_hidden:
            return logits_initial, logits_final, logits_tone, char_out
        return logits_initial, logits_final, logits_tone

In [15]:
vocab_size_tokens = len(tokenizer.vocab)
embedding_dim = 100
hidden_dim = 512
mlp_hidden_dim = 256
num_initials = len(init2idx)
num_finals   = len(final2idx)
num_tones    = len(tone2idx)
num_layers = 4

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Recreate model instance
encoder = Encoder(
    vocab_size_tokens=vocab_size_tokens,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    mlp_hidden_dim=mlp_hidden_dim,
    num_initials=num_initials,
    num_finals=num_finals,
    num_tones=num_tones,
    num_layers=num_layers
).to(device)

## Decoder Model

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Decoder(nn.Module):
    """
    Decodes character‑level hidden representations (from an encoder) into character indices.
    Optionally applies a BiLSTM on top of the input hidden states.
    """
    def __init__(self, input_dim, hidden_dim, num_layers, vocab_size_char, mlp_hidden_dim=128, dropout=0.3):
        """
        Args:
            input_dim: dimension of input hidden states (e.g., encoder hidden_dim * 2)
            hidden_dim: LSTM hidden size
            num_layers: number of LSTM layers
            vocab_size_char: size of character vocabulary
            mlp_hidden_dim: hidden dimensionality of MLP classfication head
            dropout: dropout probability (applied after LSTM)
        """
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            bidirectional=True, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, vocab_size_char, dropout)

    def forward(self, hidden_states, char_lengths):
        """
        Args:
            hidden_states: (batch, max_chars, input_dim) - padded character‑level representations
            char_lengths: (batch,) - actual lengths of each sequence (for packing)
        Returns:
            logits: (batch, max_chars, vocab_size_char) - unnormalized scores for each character
        """
        # Pack for variable‑length sequences
        packed = pack_padded_sequence(
            hidden_states, char_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        lstm_out = self.dropout(lstm_out)
        logits = self.head(lstm_out)
        return logits

In [17]:
input_dim = hidden_dim * 2
mlp_hidden_dim = 256
num_layers = 4
vocab_size_char = len(idx2char)


device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


decoder = Decoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    vocab_size_char=vocab_size_char,
    mlp_hidden_dim=mlp_hidden_dim
).to(device)

## Pipeline Model

Combine the models in the pipeline:

`character input -> encoder -> latent space -> decoder -> character output`

The training is performed in the curriculum learning fashion:
1. Train for several epochs only Encoder on its task.
2. Train for several epochs Encoder+Decoder pipeline with frozen Encoder.
3. Train for several epochs Encoder+Decoder end-to-end (with combined loss).

Additionally, I propose to consider Encoder+Decoder pipeline trained end-to-end with combined loss from scratch.

At each stage, save the best model/-s.

In [18]:
import torch
import torch.nn as nn

class EncoderDecoderPipeline(nn.Module):
    """
    Combines a character‑to‑pinyin encoder and a pinyin‑to‑character decoder.
    The encoder is expected to output character‑level hidden states (with `return_hidden=True`).
    The decoder takes those hidden states and predicts character indices.
    """
    def __init__(self, encoder, decoder, freeze_encoder=False):
        """
        Args:
            encoder: a trained/frozen encoder (e.g., CharBiLSTMForPinyin)
            decoder: a decoder (e.g., Pinyin2CharDecoder)
            freeze_encoder: if True, encoder parameters are frozen (no gradients).
        """
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, flat_tokens, char_offsets, char_lengths, return_encoder_output=False):
        """
        Args:
            flat_tokens: (total_tokens,) flattened token IDs
            char_offsets: (total_chars,) start indices for each character
            char_lengths: (batch,) number of characters per sample
        Returns:
            logits: (batch, max_chars, vocab_size_char) from decoder
        """
        # Encoder forward (returns hidden states when return_hidden=True)
        logits_initial, logits_final, logits_tone, hidden = self.encoder(flat_tokens, char_offsets, char_lengths, return_hidden=True)
        # hidden shape: (batch, max_chars, encoder_hidden_dim*2)

        # Decoder forward
        logits = self.decoder(hidden, char_lengths)

        if return_encoder_output:
            return logits_initial, logits_final, logits_tone, logits
        return logits

    def get_hidden(self, flat_tokens, char_offsets, char_lengths):
        """Return only the hidden states from the encoder (no decoder)."""
        *_, hidden = self.encoder(flat_tokens, char_offsets, char_lengths, return_hidden=True)
        return hidden

# Additional Stage: Pipeline End-to-End from Scratch

## Training Loop

In [19]:
# --- Hyperparameters for joint training from scratch ---

encoder_lr = 1e-3
decoder_lr = 1e-3
weight_decay = 1e-4
lambda_pinyin = 0.5        # weight for pinyin loss


# Pipeline (both trainable)
pipeline = EncoderDecoderPipeline(encoder, decoder, freeze_encoder=False)
pipeline.to(device)

# Optimizer with separate parameter groups
optimizer = torch.optim.AdamW([
    {'params': pipeline.encoder.parameters(), 'lr': encoder_lr, 'weight_decay': weight_decay},
    {'params': pipeline.decoder.parameters(), 'lr': decoder_lr, 'weight_decay': weight_decay}
])

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Loss functions (ignore padding)
criterion_pinyin_initial = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
criterion_pinyin_final   = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
criterion_pinyin_tone    = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
criterion_char = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Training settings
num_epochs = 20
patience = 5
best_val_loss = float('inf')
counter = 0

In [20]:
import pickle
from tqdm import tqdm

# Training loop with logging (including encoder accuracies)
best_val_loss = float('inf')
counter = 0
metrics_log = []   # list of dicts: epoch, train_loss, val_loss, val_char_acc,
                   # val_init_acc, val_final_acc, val_tone_acc, val_exact_acc

for epoch in range(num_epochs):
    # --- Training ---
    pipeline.train()
    total_loss = 0
    train_steps = 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch in train_pbar:
        # Unpack batch: flat_tokens, char_offsets, char_lengths, padded_pinyin, chars_list, padded_char_idxs
        flat_tokens, char_offsets, char_lengths, padded_pinyin, _, padded_char_idxs = batch
        flat_tokens = flat_tokens.to(device)
        char_offsets = char_offsets.to(device)
        char_lengths = char_lengths.to(device)
        padded_pinyin = padded_pinyin.to(device)
        padded_char_idxs = padded_char_idxs.to(device)

        # Forward with encoder output
        logits_i, logits_f, logits_t, char_logits = pipeline(
            flat_tokens, char_offsets, char_lengths, return_encoder_output=True
        )

        # Pinyin loss (encoder)
        loss_i = criterion_pinyin_initial(logits_i.reshape(-1, num_initials), padded_pinyin[..., 0].reshape(-1))
        loss_f = criterion_pinyin_final(logits_f.reshape(-1, num_finals), padded_pinyin[..., 1].reshape(-1))
        loss_t = criterion_pinyin_tone(logits_t.reshape(-1, num_tones), padded_pinyin[..., 2].reshape(-1))
        pinyin_loss = loss_i + loss_f + loss_t

        # Character loss (decoder)
        char_loss = criterion_char(char_logits.reshape(-1, vocab_size_char), padded_char_idxs.reshape(-1))

        # Combined loss
        loss = char_loss + lambda_pinyin * pinyin_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        train_steps += 1
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = total_loss / train_steps

    # --- Validation (compute both decoder and encoder accuracies) ---
    pipeline.eval()
    total_val_loss = 0
    total_val_char_correct = 0
    total_val_chars = 0

    # Encoder accuracy counters
    total_init_correct = 0
    total_final_correct = 0
    total_tone_correct = 0
    total_exact_correct = 0
    total_enc_chars = 0

    val_steps = 0
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for batch in val_pbar:
            flat_tokens, char_offsets, char_lengths, padded_pinyin, _, padded_char_idxs = batch
            flat_tokens = flat_tokens.to(device)
            char_offsets = char_offsets.to(device)
            char_lengths = char_lengths.to(device)
            padded_pinyin = padded_pinyin.to(device)
            padded_char_idxs = padded_char_idxs.to(device)

            logits_i, logits_f, logits_t, char_logits = pipeline(
                flat_tokens, char_offsets, char_lengths, return_encoder_output=True
            )

            # Losses
            loss_i = criterion_pinyin_initial(logits_i.reshape(-1, num_initials), padded_pinyin[..., 0].reshape(-1))
            loss_f = criterion_pinyin_final(logits_f.reshape(-1, num_finals), padded_pinyin[..., 1].reshape(-1))
            loss_t = criterion_pinyin_tone(logits_t.reshape(-1, num_tones), padded_pinyin[..., 2].reshape(-1))
            pinyin_loss = loss_i + loss_f + loss_t
            char_loss = criterion_char(char_logits.reshape(-1, vocab_size_char), padded_char_idxs.reshape(-1))
            val_loss = char_loss + lambda_pinyin * pinyin_loss

            total_val_loss += val_loss.item()
            val_steps += 1

            # Decoder character accuracy
            preds = char_logits.argmax(dim=-1)
            mask_char = padded_char_idxs != PAD_IDX
            correct_char = (preds == padded_char_idxs) & mask_char
            total_val_char_correct += correct_char.sum().item()
            total_val_chars += mask_char.sum().item()

            # Encoder accuracy (initial, final, tone, exact)
            mask_enc = (padded_pinyin[..., 0] != PAD_IDX)   # same for all components
            pred_i = logits_i.argmax(dim=-1)
            pred_f = logits_f.argmax(dim=-1)
            pred_t = logits_t.argmax(dim=-1)
            total_init_correct += ((pred_i == padded_pinyin[..., 0]) & mask_enc).sum().item()
            total_final_correct += ((pred_f == padded_pinyin[..., 1]) & mask_enc).sum().item()
            total_tone_correct += ((pred_t == padded_pinyin[..., 2]) & mask_enc).sum().item()
            exact = (pred_i == padded_pinyin[..., 0]) & (pred_f == padded_pinyin[..., 1]) & (pred_t == padded_pinyin[..., 2]) & mask_enc
            total_exact_correct += exact.sum().item()
            total_enc_chars += mask_enc.sum().item()

            val_pbar.set_postfix({'loss': f'{val_loss.item():.4f}'})

    avg_val_loss = total_val_loss / val_steps
    val_char_acc = total_val_char_correct / total_val_chars if total_val_chars > 0 else 0
    val_init_acc = total_init_correct / total_enc_chars if total_enc_chars > 0 else 0
    val_final_acc = total_final_correct / total_enc_chars if total_enc_chars > 0 else 0
    val_tone_acc = total_tone_correct / total_enc_chars if total_enc_chars > 0 else 0
    val_exact_acc = total_exact_correct / total_enc_chars if total_enc_chars > 0 else 0

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")
    print(f"  Decoder Char Acc: {val_char_acc:.4f}")
    print(f"  Encoder: Init={val_init_acc:.4f}, Final={val_final_acc:.4f}, Tone={val_tone_acc:.4f}, Exact={val_exact_acc:.4f}")

    # Log metrics
    metrics_log.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'val_char_acc': val_char_acc,
        'val_init_acc': val_init_acc,
        'val_final_acc': val_final_acc,
        'val_tone_acc': val_tone_acc,
        'val_exact_acc': val_exact_acc
    })

    # Early stopping and best model saving (based on validation loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(pipeline.state_dict(), 'pipeline_scratch_best.pt')
        print(f"  -> Best pipeline saved (val loss: {best_val_loss:.4f})")
        counter = 0
    else:
        counter += 1
        print(f"  -> No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    # Step scheduler with validation loss
    scheduler.step(avg_val_loss)

print("Training from scratch complete.")

# Save metrics log
with open('scratch_metrics.pkl', 'wb') as f:
    pickle.dump(metrics_log, f)

Epoch 1/20 [Val]: 100%|██████████| 505/505 [00:30<00:00, 16.55it/s, loss=0.8012]


Epoch 1: Train Loss = 3.2006, Val Loss = 0.8389
  Decoder Char Acc: 0.8652
  Encoder: Init=0.9370, Final=0.9352, Tone=0.9537, Exact=0.9112
  -> Best pipeline saved (val loss: 0.8389)


Epoch 2/20 [Val]: 100%|██████████| 505/505 [00:30<00:00, 16.55it/s, loss=0.2921]


Epoch 2: Train Loss = 0.7759, Val Loss = 0.2899
  Decoder Char Acc: 0.9656
  Encoder: Init=0.9664, Final=0.9652, Tone=0.9729, Exact=0.9515
  -> Best pipeline saved (val loss: 0.2899)


Epoch 3/20 [Val]: 100%|██████████| 505/505 [00:30<00:00, 16.56it/s, loss=0.1904]


...

Epoch 18: Train Loss = 0.1140, Val Loss = 0.0941
  Decoder Char Acc: 0.9936
  Encoder: Init=0.9884, Final=0.9876, Tone=0.9895, Exact=0.9829
  -> Best pipeline saved (val loss: 0.0941)


Epoch 19/20 [Val]: 100%|██████████| 505/505 [00:30<00:00, 16.55it/s, loss=0.0843]


Epoch 19: Train Loss = 0.1120, Val Loss = 0.0942
  Decoder Char Acc: 0.9937
  Encoder: Init=0.9884, Final=0.9877, Tone=0.9897, Exact=0.9831
  -> No improvement for 1 epoch(s).


Epoch 20/20 [Val]: 100%|██████████| 505/505 [00:30<00:00, 16.59it/s, loss=0.1076]

Epoch 20: Train Loss = 0.1102, Val Loss = 0.0949
  Decoder Char Acc: 0.9938
  Encoder: Init=0.9883, Final=0.9877, Tone=0.9897, Exact=0.9831
  -> No improvement for 2 epoch(s).
Training from scratch complete.


## Evaluate on Test Set

In [21]:
def evaluate_pipeline(pipeline, dataloader, device, ignore_index=-100):
    """
    Evaluate the pipeline on a dataloader.
    Returns accuracy over non‑padded characters.
    """
    pipeline.eval()
    total_correct = 0
    total_chars = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluation'):
            flat_tokens, char_offsets, char_lengths, _, _, padded_char_idxs = batch
            flat_tokens = flat_tokens.to(device)
            char_offsets = char_offsets.to(device)
            char_lengths = char_lengths.to(device)
            padded_char_idxs = padded_char_idxs.to(device)

            logits = pipeline(flat_tokens, char_offsets, char_lengths)
            preds = logits.argmax(dim=-1)

            mask = padded_char_idxs != ignore_index
            correct = (preds == padded_char_idxs) & mask
            total_correct += correct.sum().item()
            total_chars += mask.sum().item()

    accuracy = total_correct / total_chars if total_chars > 0 else 0
    return accuracy

In [22]:
pipeline.load_state_dict(torch.load('pipeline_scratch_best.pt', map_location=device))
pipeline.eval()

print("Pipeline loaded successfully for evaluation.")

Pipeline loaded successfully for evaluation.


In [23]:
test_acc = evaluate_pipeline(pipeline, test_loader, device, ignore_index=PAD_IDX)
print(f"Final test accuracy: {test_acc:.4f}")

Evaluation: 100%|██████████| 505/505 [00:29<00:00, 17.02it/s]

Final test accuracy: 0.9936
